# 03 — what resists?

**triplet-phase-lock**

This notebook introduces the resistance stage. We align triplet states to a reference direction, apply a 45° cosine threshold, and compare clean, weakly perturbed, strongly perturbed, and noisy trajectories.

Guiding question:

- **Π (resist): what resists?**


## Setup

We continue from the expand and extend stages.

Triplets are still defined as:

\\[\nT_k = (N_k, N_{k+1}, N_{k+2})\n\\]\n
Resistance is measured by alignment to a reference direction using cosine similarity:

\\[\n\\cos\\theta_k = \\frac{T_k \\cdot R}{\\|T_k\\|\\,\\|R\\|}\n\\]\n
We then apply the 45° threshold:

\\[\n\\cos\\theta_k \\ge \\frac{1}{\\sqrt{1^2 + 1^2}}\n\\]\n
This notebook studies how strongly each sequence resists misalignment under that constraint.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams['figure.figsize'] = (8, 4)
plt.rcParams['axes.grid'] = True


## Sequence and triplet helpers


In [ ]:
def sequence_n(k):
    k = np.asarray(k, dtype=float)
    return 24.0 * k - 25.0


def sequence_n_perturbed(k, amplitude=8.0, period=6.0):
    k = np.asarray(k, dtype=float)
    return sequence_n(k) + amplitude * np.sin(k / period)


def sequence_n_noisy(k, amplitude=40.0, seed=7):
    """
    A more disruptive comparison sequence:
    arithmetic trend + Gaussian noise
    """
    k = np.asarray(k, dtype=float)
    rng = np.random.default_rng(seed)
    return sequence_n(k) + rng.normal(loc=0.0, scale=amplitude, size=len(k))


def build_triplets_from_values(vals):
    vals = np.asarray(vals, dtype=float)
    return np.stack([vals[:-2], vals[1:-1], vals[2:]], axis=1)


def normalize_rows(x, eps=1e-12):
    norms = np.linalg.norm(x, axis=1, keepdims=True)
    return x / np.maximum(norms, eps)


def cosine_scores(x, reference):
    x_norm = normalize_rows(x)
    r = np.asarray(reference, dtype=float)
    r = r / max(np.linalg.norm(r), 1e-12)
    return x_norm @ r


def resistance_mask(x, reference, threshold):
    return cosine_scores(x, reference) >= threshold


def resistance_ratio(mask):
    mask = np.asarray(mask, dtype=bool)
    if mask.size == 0:
        return 0.0
    return float(mask.mean())


def resistance_margin(scores, threshold):
    return scores - threshold


def local_differences(x):
    if len(x) < 2:
        return np.array([], dtype=float)
    return np.linalg.norm(np.diff(x, axis=0), axis=1)


## Generate clean, weak, strong, and noisy triplets


In [ ]:
num_steps = 80
k = np.arange(1, num_steps + 3)

clean_vals = sequence_n(k)
weak_vals = sequence_n_perturbed(k, amplitude=8.0, period=6.0)
strong_vals = sequence_n_perturbed(k, amplitude=20.0, period=4.0)
noisy_vals = sequence_n_noisy(k, amplitude=40.0, seed=7)

triplets_clean = build_triplets_from_values(clean_vals)
triplets_weak = build_triplets_from_values(weak_vals)
triplets_strong = build_triplets_from_values(strong_vals)
triplets_noisy = build_triplets_from_values(noisy_vals)

print('Triplet shapes:')
print('clean:', triplets_clean.shape)
print('weak :', triplets_weak.shape)
print('strong:', triplets_strong.shape)
print('noisy:', triplets_noisy.shape)


## Reference direction and threshold

A natural balanced reference is:

\\[\nR = (1,1,1)\n\\]\n
and the 45° threshold is:

\\[\n\\frac{1}{\\sqrt{1^2 + 1^2}}\n\\]\n

In [ ]:
reference = np.array([1.0, 1.0, 1.0])
threshold_45 = 1.0 / np.sqrt(1.0**2 + 1.0**2)

print(f'Reference vector: {reference}')
print(f'45° threshold: {threshold_45:.12f}')


## Plot 1 — raw sequence comparison

Start with a direct view of the four sequence families.


In [ ]:
plt.figure()
plt.plot(k, clean_vals, marker='o', label='clean')
plt.plot(k, weak_vals, marker='o', label='weak perturbation')
plt.plot(k, strong_vals, marker='o', label='strong perturbation')
plt.plot(k, noisy_vals, marker='o', label='noisy')
plt.xlabel('k')
plt.ylabel('value')
plt.title('Sequence comparison at the resistance stage')
plt.legend()
plt.tight_layout()
plt.show()


## Resistance scores

Compute cosine alignment for each triplet family.


In [ ]:
scores_clean = cosine_scores(triplets_clean, reference)
scores_weak = cosine_scores(triplets_weak, reference)
scores_strong = cosine_scores(triplets_strong, reference)
scores_noisy = cosine_scores(triplets_noisy, reference)

print(f'Clean score range : {scores_clean.min():.12f} to {scores_clean.max():.12f}')
print(f'Weak score range  : {scores_weak.min():.12f} to {scores_weak.max():.12f}')
print(f'Strong score range: {scores_strong.min():.12f} to {scores_strong.max():.12f}')
print(f'Noisy score range : {scores_noisy.min():.12f} to {scores_noisy.max():.12f}')


In [ ]:
score_idx = np.arange(1, len(scores_clean) + 1)

plt.figure()
plt.plot(score_idx, scores_clean, marker='o', label='clean')
plt.plot(score_idx, scores_weak, marker='o', label='weak perturbation')
plt.plot(score_idx, scores_strong, marker='o', label='strong perturbation')
plt.plot(score_idx, scores_noisy, marker='o', label='noisy')
plt.axhline(threshold_45, linestyle='--', label='45° threshold')
plt.xlabel('k')
plt.ylabel(r'$\cos\theta_k$')
plt.title('Resistance scores relative to the reference direction')
plt.legend()
plt.tight_layout()
plt.show()


## Acceptance masks and resistance ratios

Apply the 45° threshold and measure the fraction of triplets that resist misalignment.


In [ ]:
mask_clean = resistance_mask(triplets_clean, reference, threshold_45)
mask_weak = resistance_mask(triplets_weak, reference, threshold_45)
mask_strong = resistance_mask(triplets_strong, reference, threshold_45)
mask_noisy = resistance_mask(triplets_noisy, reference, threshold_45)

print(f'Clean resistance ratio : {resistance_ratio(mask_clean):.6f}')
print(f'Weak resistance ratio  : {resistance_ratio(mask_weak):.6f}')
print(f'Strong resistance ratio: {resistance_ratio(mask_strong):.6f}')
print(f'Noisy resistance ratio : {resistance_ratio(mask_noisy):.6f}')


In [ ]:
plt.figure()
plt.plot(score_idx, mask_clean.astype(float), marker='o', label='clean')
plt.plot(score_idx, mask_weak.astype(float), marker='o', label='weak perturbation')
plt.plot(score_idx, mask_strong.astype(float), marker='o', label='strong perturbation')
plt.plot(score_idx, mask_noisy.astype(float), marker='o', label='noisy')
plt.xlabel('k')
plt.ylabel('accepted (1) / rejected (0)')
plt.title('Resistance masks under the 45° threshold')
plt.legend()
plt.tight_layout()
plt.show()


## Resistance margins

Measure how far above or below threshold each triplet sits:

\\[\nm_k = \\cos\\theta_k - \\frac{1}{\\sqrt{1^2 + 1^2}}\n\\]\n

In [ ]:
margin_clean = resistance_margin(scores_clean, threshold_45)
margin_weak = resistance_margin(scores_weak, threshold_45)
margin_strong = resistance_margin(scores_strong, threshold_45)
margin_noisy = resistance_margin(scores_noisy, threshold_45)

print(f'Clean margin range : {margin_clean.min():.12f} to {margin_clean.max():.12f}')
print(f'Weak margin range  : {margin_weak.min():.12f} to {margin_weak.max():.12f}')
print(f'Strong margin range: {margin_strong.min():.12f} to {margin_strong.max():.12f}')
print(f'Noisy margin range : {margin_noisy.min():.12f} to {margin_noisy.max():.12f}')


In [ ]:
plt.figure()
plt.plot(score_idx, margin_clean, marker='o', label='clean')
plt.plot(score_idx, margin_weak, marker='o', label='weak perturbation')
plt.plot(score_idx, margin_strong, marker='o', label='strong perturbation')
plt.plot(score_idx, margin_noisy, marker='o', label='noisy')
plt.axhline(0.0, linestyle='--', label='threshold boundary')
plt.xlabel('k')
plt.ylabel('score minus threshold')
plt.title('Resistance margins')
plt.legend()
plt.tight_layout()
plt.show()


## Accepted triplet-path comparison

Project only accepted triplets into the first two coordinates.

This gives a geometric view of what remains inside the resistance constraint.


In [ ]:
plt.figure()
plt.plot(triplets_clean[mask_clean, 0], triplets_clean[mask_clean, 1], marker='o', label='clean accepted')
plt.plot(triplets_weak[mask_weak, 0], triplets_weak[mask_weak, 1], marker='o', label='weak accepted')
plt.plot(triplets_strong[mask_strong, 0], triplets_strong[mask_strong, 1], marker='o', label='strong accepted')
plt.plot(triplets_noisy[mask_noisy, 0], triplets_noisy[mask_noisy, 1], marker='o', label='noisy accepted')
plt.xlabel(r'$N_k$')
plt.ylabel(r'$N_{k+1}$')
plt.title('Accepted triplet paths under the resistance constraint')
plt.legend()
plt.tight_layout()
plt.show()


## Local variation among accepted triplets

Measure how much accepted structure still varies locally.


In [ ]:
accepted_delta_clean = local_differences(triplets_clean[mask_clean])
accepted_delta_weak = local_differences(triplets_weak[mask_weak])
accepted_delta_strong = local_differences(triplets_strong[mask_strong])
accepted_delta_noisy = local_differences(triplets_noisy[mask_noisy])

print(f'Accepted clean Δ range : {accepted_delta_clean.min():.12f} to {accepted_delta_clean.max():.12f}')
print(f'Accepted weak Δ range  : {accepted_delta_weak.min():.12f} to {accepted_delta_weak.max():.12f}')
print(f'Accepted strong Δ range: {accepted_delta_strong.min():.12f} to {accepted_delta_strong.max():.12f}')
print(f'Accepted noisy Δ range : {accepted_delta_noisy.min():.12f} to {accepted_delta_noisy.max():.12f}')


In [ ]:
plt.figure()
plt.plot(np.arange(1, len(accepted_delta_clean) + 1), accepted_delta_clean, marker='o', label='clean accepted Δ')
plt.plot(np.arange(1, len(accepted_delta_weak) + 1), accepted_delta_weak, marker='o', label='weak accepted Δ')
plt.plot(np.arange(1, len(accepted_delta_strong) + 1), accepted_delta_strong, marker='o', label='strong accepted Δ')
plt.plot(np.arange(1, len(accepted_delta_noisy) + 1), accepted_delta_noisy, marker='o', label='noisy accepted Δ')
plt.xlabel('accepted-step index')
plt.ylabel(r'$\Delta$ among accepted triplets')
plt.title('Local variation after resistance filtering')
plt.legend()
plt.tight_layout()
plt.show()


## Summary

This notebook established the **resist** stage:

- defined a reference direction
- computed cosine alignment scores
- applied the 45° threshold
- measured resistance masks and ratios
- quantified resistance margins
- visualized accepted triplet paths

Next notebook:

- compare the full pipeline across stages
- summarize expand → extend → resist
- connect the sequence families in one cross-stage view
